# Train a multi-turn Wordle agent with GRPO on OpenEnv using Hugging Face Jobs

_Authored by: [Behrooz Azarkhalili](https://github.com/behroozazarkhalili)_

**Agent RL is awkward to run in a notebook.** GRPO holds the policy *and* generates the
rollouts, so a real run is hours of GPU time — too long for an interactive session, and it
dies with your kernel. This recipe shows the alternative: **submit the notebook itself to
[Hugging Face Jobs](https://huggingface.co/docs/huggingface_hub/en/guides/cli#hf-jobs) and
let [`papermill`](https://papermill.readthedocs.io/) execute it non-interactively on HF's
cloud.** You get the notebook's readability with a batch job's durability, and the executed
notebook — outputs and all — comes back as an artifact.

Multi-turn Wordle is the worked example: it runs on the official
[`openenv/wordle`](https://huggingface.co/spaces/openenv/wordle) Space and needs no external
API keys.

> **Attribution.** The environment wrapper and training setup build directly on TRL's official
> Wordle example ([`openenv_wordle_grpo.ipynb`](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb)
> and the [TRL OpenEnv guide](https://huggingface.co/docs/trl/en/openenv)). **What this recipe
> adds is the non-interactive Jobs workflow around it** — papermill submission, `SMOKE` gating,
> compute auto-detection, throughput calibration, and the environment-Space operational details
> below.

### What you'll learn

- Submit a notebook to run **non-interactively on Hugging Face Jobs** with `papermill` — the GPU work runs on HF's cloud, you only submit
- Gate every expensive quantity behind a `SMOKE` switch so you prove the pipeline end-to-end in minutes before paying for a full run
- **Calibrate the full run from a measurement** — time the smoke rollouts, project the real run, and size `--flavor`/`--timeout`/steps from your own hardware instead of a guess
- Wrap a **multi-turn, stateful** [OpenEnv](https://github.com/huggingface/OpenEnv) environment where each rollout spans up to 6 `guess` tool calls and the *environment* decides when the episode is `done`
- Keep **training and evaluation prompts identical** by deriving the tool schema from the environment class the same way TRL does
- Evaluate a multi-turn agent the faithful way — by **playing complete games** and measuring win rate

> **New to OpenEnv?** For the single-turn case and the SFT-warm-start-then-GRPO story, see the
> OpenEnv [SFT-warmup tutorial](https://huggingface.co/docs/openenv/tutorials/sft-warmup) and the
> [end-to-end walkthrough](https://huggingface.co/docs/openenv/tutorials/end-to-end-walkthrough).

## ▶ Submit this notebook as an HF Job

Requires a positive [credit balance](https://huggingface.co/settings/billing) (Jobs are
pay-as-you-go — you pay only for the seconds you use). The control plane is plain HTTPS; the
recipe below clones this notebook straight from the cookbook repo, so it runs as-is.

**Start here — the smoke run.** It exercises every cell end-to-end in minutes for a few cents,
and prints the throughput measurement you need to size the real run.

```bash
hf jobs run \
  --flavor a10g-small \
  --timeout 3600 \
  --secrets HF_TOKEN \
  -e SMOKE=1 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=https://openenv-wordle.hf.space \
  python:3.12 \
  bash -c "apt-get update -q && apt-get install -y -q git && \
           pip install -q papermill ipykernel && \
           python -m ipykernel install --user --name python3 && \
           pip install -q trl openenv 'transformers>=5.3.0' trackio jmespath nest_asyncio datasets && \
           GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://huggingface.co/spaces/openenv/wordle /wordle_env && \
           pip install -q --no-deps /wordle_env && \
           git clone --depth 1 https://github.com/huggingface/cookbook /cookbook && cd /cookbook/notebooks/en && \
           papermill grpo_agent_wordle_hf_jobs.ipynb out.ipynb"
```

`--secrets HF_TOKEN` forwards your token so the trained model can push to the Hub.

### Then: the full run

The smoke run's final cell prints a **measured** projection — seconds per rollout on your GPU
and model, and the wall-clock the configured `SMOKE=0` settings imply. **Use those numbers to
set `--flavor`, `--timeout`, and `GRPO_MAX_STEPS`**, rather than trusting the defaults below.
Then re-submit with `-e SMOKE=0` and `-e ENV_BASE_URL` pointing at *your own* environment Space
(see the next section — this is required, not optional).

```bash
hf jobs run \
  --flavor a10g-small \
  --timeout 10800 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e REPORT_TO=trackio \
  -e GRPO_MAX_STEPS=<from the calibration cell> \
  -e ENV_BASE_URL=https://<your-username>-wordle.hf.space \
  python:3.12 \
  bash -c "apt-get update -q && apt-get install -y -q git && \
           pip install -q papermill ipykernel && \
           python -m ipykernel install --user --name python3 && \
           pip install -q trl openenv 'transformers>=5.3.0' trackio jmespath nest_asyncio datasets && \
           GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://huggingface.co/spaces/openenv/wordle /wordle_env && \
           pip install -q --no-deps /wordle_env && \
           git clone --depth 1 https://github.com/huggingface/cookbook /cookbook && cd /cookbook/notebooks/en && \
           papermill grpo_agent_wordle_hf_jobs.ipynb out.ipynb"
```

### ⚠ Use your own environment Space for training runs

The Wordle environment is **single-session by design**, and the limit is not a tuning knob:

- `TextArenaEnvironment` inherits `SUPPORTS_CONCURRENT_SESSIONS = False` from OpenEnv's `Environment` base class
- the Space builds its app without a concurrency override, so the server defaults to `max_concurrent_envs=1`
- OpenEnv *refuses to start* a server with `max_concurrent_envs > 1` for an environment not marked concurrent, so you cannot simply raise it

Consequences worth knowing before you submit a long job:

1. **The shared `openenv/wordle` Space serves one session at a time, community-wide.** Fine for the
   smoke run above; not something to occupy for hours.
2. **A crashed run leaks its session.** `reset()` opens a session and a crash never calls `close()`,
   so failed attempts pile up until new connections are refused with
   `Server at capacity: 1/1 sessions active (code: CAPACITY_REACHED)`.
3. **Owning the Space is what lets you recover from (2)** — you can restart it and clear leaked
   sessions. You cannot restart a Space you do not own.

So duplicate it once, and point `ENV_BASE_URL` at your copy:

```python
from huggingface_hub import HfApi
api = HfApi()
api.duplicate_space("openenv/wordle")            # -> <your-username>/wordle
# If a crashed run left it wedged, this clears every server-side session:
# api.restart_space("<your-username>/wordle")
```

Your Space URL follows the pattern `https://<owner>-<space-name>.hf.space`, lowercased with
`.`/`_` replaced by `-`.

### Faster rollouts with vLLM

An HF Job is a non-interactive runtime, so vLLM-accelerated generation works here (it is left off
in interactive notebooks because its init conflicts with IPython). Use a bigger GPU and install
vLLM in the same command — note it still needs `ipykernel` registered, since papermill runs the
notebook through a kernel either way:

```bash
hf jobs run \
  --flavor a100-large \
  --timeout 21600 \
  --secrets HF_TOKEN \
  -e SMOKE=0 -e USE_VLLM=1 -e REPORT_TO=trackio \
  -e ENV_BASE_URL=https://<your-username>-wordle.hf.space \
  python:3.12 \
  bash -c "apt-get update -q && apt-get install -y -q git && \
           pip install -q papermill ipykernel vllm && \
           python -m ipykernel install --user --name python3 && \
           pip install -q trl openenv 'transformers>=5.3.0' trackio jmespath nest_asyncio datasets && \
           GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://huggingface.co/spaces/openenv/wordle /wordle_env && \
           pip install -q --no-deps /wordle_env && \
           git clone --depth 1 https://github.com/huggingface/cookbook /cookbook && cd /cookbook/notebooks/en && \
           papermill grpo_agent_wordle_hf_jobs.ipynb out.ipynb"
```

## 0 · Install

We install TRL (the trainer), OpenEnv (the env client), and the environment's own package
(the Wordle env client ships from the `openenv/wordle` Space). `transformers>=5.3.0` is
required because GRPO's `environment_factory` path — a TRL feature — depends on
tool-calling / chat-template behavior introduced in that Transformers release. vLLM stays
off in-notebook (its init conflicts with IPython) and is enabled only via `USE_VLLM=1` on
an HF Job, where a non-interactive runtime makes it safe.

The install helper is **idempotent**: re-running this cell is a no-op, and so is running it
on a Job whose `bash -c` already installed the environment package. Its docstring records
why the env Space is cloned in full rather than installed with `pip install git+…`.

In [ ]:
%pip install -q trl openenv "transformers>=5.3.0" trackio jmespath nest_asyncio datasets

import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def ensure_env_package(space_id: str, module: str, workdir: str = "/tmp/openenv_spaces") -> None:
    """Install an OpenEnv environment client from its Hub Space, idempotently.

    Re-running this cell is a no-op, and so is running it on an HF Job whose
    `bash -c` already installed the package: if `module` imports, we return early.

    We clone in full rather than letting pip do `git+https://...`, because pip
    uses a partial ("promisor") clone and some Space repos are not served
    reliably that way. Verified 2026-08-02:

        $ git clone --filter=blob:none .../spaces/openenv/wordle
        fatal: expected 'packfile'
        fatal: could not fetch <sha> from promisor remote      # empty worktree

    while the identical command against .../spaces/sergiopaniego/reasoning_gym
    exits 0. So this is per-Space behaviour, not a general property of HF Spaces
    -- a full clone simply works for both. LFS blobs are skipped: environment
    clients are pure Python, with no model weights to fetch.
    """
    if importlib.util.find_spec(module) is not None:
        print(f"'{module}' already importable — skipping install.")
        return

    dest = Path(workdir) / space_id.replace("/", "__")
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--depth", "1",
             f"https://huggingface.co/spaces/{space_id}", str(dest)],
            check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(dest)], check=True)
    print(f"Installed '{module}' from Space '{space_id}'.")


# Both are env-overridable, so pointing this notebook at a different OpenEnv
# environment needs no code edit.
ENV_SPACE_ID = os.environ.get("ENV_SPACE_ID", "openenv/wordle")
ENV_MODULE = os.environ.get("ENV_MODULE", "textarena_env")
ensure_env_package(ENV_SPACE_ID, module=ENV_MODULE)

### Why `nest_asyncio`?

The OpenEnv client is **async** (`await env.reset()`, `await env.step()`), but a Jupyter/Colab kernel already runs its own event loop. `nest_asyncio.apply()` lets us `await` inside notebook cells without `RuntimeError: event loop already running`.

> 💡 In a plain `.py` script you'd use `asyncio.run(...)` instead — this shim is specifically for notebook runtimes.

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
# --- Authenticate with the Hugging Face Hub (portable) -------------------------
# Prefers an already-set HF_TOKEN (HF Jobs / Colab secret); falls back to the
# interactive widget. Never hard-codes a token.
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login

    login(token=os.environ["HF_TOKEN"])
    print("Authenticated via HF_TOKEN.")
else:
    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:  # non-interactive without a token
        print(f"notebook_login unavailable ({exc}); set HF_TOKEN before running.")

### Resolve your Hub username

Downstream repo names (rollouts dataset, trained model) are built from your username, so we resolve it once via `whoami()` rather than hard-coding it. This keeps the notebook portable across accounts.

> 💡 `whoami()` reads the token you authenticated with above — no extra prompt.

In [ ]:
# Resolve your Hub username automatically — no hard-coded usernames downstream.
from huggingface_hub import whoami

try:
    HF_USERNAME = whoami()["name"]
    print(f"Hub user: {HF_USERNAME}")
except Exception as exc:
    HF_USERNAME = os.environ.get("HF_USERNAME", "")
    print(f"Could not resolve whoami() ({exc}); using HF_USERNAME={HF_USERNAME!r}.")

### Auto-detect compute, pick a model that fits

Rather than hard-code a model, we read the GPU's VRAM and choose a size that fits (`Qwen3-1.7B` when there's headroom, else `Qwen3-0.6B`). Override with `MODEL_NAME`. This is what lets the same notebook run on a T4, an L4, or an A100 unchanged.

> 💡 GRPO holds the policy model **and** generates rollouts, so VRAM headroom matters more than for plain inference. [TRL GRPO docs](https://huggingface.co/docs/trl/en/grpo_trainer).

In [ ]:
# --- Auto-detect compute + pick a model that fits ------------------------------
# Larger model only when there is VRAM headroom; otherwise fall back. You can
# override by setting MODEL_NAME in the environment before launching.
import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    DEVICE = "cuda"
else:
    vram_gb = 0.0
    DEVICE = "cpu"

DEFAULT_MODEL = "Qwen/Qwen3-1.7B" if vram_gb >= 24 else "Qwen/Qwen3-0.6B"
MODEL_NAME = os.environ.get("MODEL_NAME", DEFAULT_MODEL)
print(f"device={DEVICE}  vram={vram_gb:.1f} GB  ->  MODEL_NAME={MODEL_NAME}")

### Run knobs: smoke vs. full

Every expensive quantity (rollout count, GRPO steps, eval size) is gated by `SMOKE` so you can prove the whole pipeline end-to-end in minutes (`SMOKE=1`) before committing to a real run (`SMOKE=0`). The values come from environment variables, so you never edit cells to change a run.

> 💡 This is the single switch that turns a 5-minute demo into a real training run.

In [ ]:
def _env_flag(name: str, default: str = "0") -> bool:
    """Read a boolean knob from the environment ('0'/'false' are false)."""
    return os.environ.get(name, default) not in ("0", "false", "False")


def _env_int(name: str, default: int) -> int:
    """Read an integer knob from the environment, falling back to `default`."""
    raw = os.environ.get(name)
    return int(raw) if raw and raw.strip().lstrip("-").isdigit() else default


SMOKE = _env_flag("SMOKE", "1")
ENV_BASE_URL = os.environ.get("ENV_BASE_URL", "https://openenv-wordle.hf.space")

# Reference size of a real run. The calibration cell projects against these, and
# they are the SMOKE=0 defaults — one definition, no duplicated constants.
FULL_RUN = {"steps": 150, "grad_accum": 64}

# Every knob stays env-overridable, so the projection the calibration cell prints
# can be fed straight back in as `-e GRPO_MAX_STEPS=...` without editing a cell.
GRPO_MAX_STEPS = _env_int("GRPO_MAX_STEPS", 5 if SMOKE else FULL_RUN["steps"])
GRAD_ACCUM = _env_int("GRAD_ACCUM", 8 if SMOKE else FULL_RUN["grad_accum"])
N_EVAL_GAMES = _env_int("N_EVAL_GAMES", 3 if SMOKE else 20)
NUM_GENERATIONS = _env_int("NUM_GENERATIONS", 2)

# Single source of truth for chat-template rendering. Training passes this to
# GRPOConfig and evaluation passes the SAME dict to apply_chat_template, so the
# two can never drift apart.
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}

print(
    f"SMOKE={SMOKE}  grpo_steps={GRPO_MAX_STEPS}  grad_accum={GRAD_ACCUM}  "
    f"num_generations={NUM_GENERATIONS}  eval_games={N_EVAL_GAMES}\nenv={ENV_BASE_URL}"
)

## 1 · System prompt — teach the rules and the tool

The prompt states the Wordle rules, the feedback colour code, and — critically — that
the model must call the `guess` tool. The `environment_factory` loop drives tool calls,
so the model has to know the tool exists.

In [ ]:
WORDLE_PROMPT = """You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

Follow these rules to play Wordle:

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN (G): Letter is correct and in the correct position
   - YELLOW (Y): Letter is in the word but in the wrong position
   - GRAY (X): Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed
6. Use the tool `guess` to make a guess.
"""

## 2 · The multi-turn environment class

This wrapper follows TRL's official Wordle example. Two things make it *multi-turn*,
and both are worth seeing explicitly if you have only met single-turn OpenEnv
environments (as in the OpenEnv
[end-to-end walkthrough](https://huggingface.co/docs/openenv/tutorials/end-to-end-walkthrough)):

- **`reset()` returns the running feedback transcript**, and the env appends new feedback
  each turn. We keep `self._last_full_feedback` and slice out only the *newly appended*
  part so the model sees just the latest result.
- **`done` is set by the env**, not by us — the game ends on a win or after 6 guesses, so
  a single rollout spans multiple `guess` tool calls. The trainer keeps stepping until
  `done=True`.

`guess` is the one tool: TRL discovers it because it is a public method with a docstring.
It penalises invalid moves (reward 0) and otherwise records the env's reward.

The `env_tools()` helper at the bottom mirrors that same discovery rule so the evaluation
later can render an identical prompt — see §5.

In [ ]:
import inspect

from textarena_env import TextArenaAction, TextArenaEnv


class WordleEnv:
    """Multi-turn Wordle rollout: up to 6 `guess` tool calls until done."""

    def __init__(self):
        self.client = TextArenaEnv(base_url=ENV_BASE_URL).sync()

    def reset(self, **kwargs) -> None | str:
        result = self.client.reset()
        # Env returns cumulative feedback; store the full text so we can diff each turn.
        self._last_full_feedback = result.observation.messages[0].content
        self.reward = 0.0
        self.done = False
        return self._last_full_feedback

    def guess(self, guess: str) -> str:
        """Make a guess in the Wordle environment.

        Args:
            guess: The guessed word, formatted as '[abcde]'.

        Returns:
            The feedback message from the environment.
        """
        if self.done:
            raise ValueError("Game over.")
        result = self.client.step(TextArenaAction(message=guess))
        full = result.observation.messages[0].content
        feedback = full[len(self._last_full_feedback):]  # only the newly appended part
        self._last_full_feedback = full
        self.reward = 0.0 if "You attempted an invalid move" in feedback else result.reward
        self.done = result.done
        return feedback


def reward_func(environments, **kwargs) -> list[float]:
    return [env.reward for env in environments]


def env_tools(env) -> list:
    """The tools TRL exposes to the policy for an OpenEnv environment instance.

    `GRPOTrainer` builds its tool list by introspecting the environment instance:
    every public bound method except `reset` and `get_reward` becomes a tool, and
    it hands those *callables* straight to `apply_chat_template(tools=...)`, which
    derives the JSON schema from each signature and docstring.

    Reproducing that rule here — rather than hand-writing a schema — is what keeps
    evaluation prompts byte-identical to the ones the policy was trained on. It
    also names no tool, so swapping `WordleEnv` for any other OpenEnv wrapper
    needs no edit: the spine stays, the env changes.
    """
    return [
        method
        for name, method in inspect.getmembers(env, predicate=inspect.ismethod)
        if name not in ("reset", "get_reward") and not name.startswith("_")
    ]

## 3 · Dataset + GRPO config

The dataset is a stack of identical dummy prompts: its *length* sets the episode count,
because the real prompt for each rollout comes from `env.reset()` inside the factory.
Multi-turn games need a larger `max_completion_length` than a single-answer environment
(one completion has to hold a whole game) and a bigger `gradient_accumulation_steps`,
per TRL's Wordle example.

In [ ]:
from datasets import Dataset

N_PROMPTS = 30 if SMOKE else 3000
dataset = Dataset.from_dict(
    {"prompt": [[{"role": "user", "content": WORDLE_PROMPT}] for _ in range(N_PROMPTS)]}
)

### Configure GRPO

`GRPOConfig` holds the RL hyperparameters. The agent-specific ones: `num_generations` (rollouts sampled per prompt — GRPO ranks them against each other), `max_completion_length` (room for the tool call), and `report_to="trackio"` for live charts. `save_strategy`/`push_to_hub` control persistence.

> 💡 GRPO is *value-free*: it needs no separate reward model — the **environment's** scalar reward is the only signal. [GRPO paper](https://huggingface.co/papers/2402.03300).

In [ ]:
import re as _re

from trl import GRPOConfig

# Hyphen-lowercase slug for clean HF/trackio Space ids.
GRPO_OUT = _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9]+", "-", f"wordle-grpo-{MODEL_NAME.split('/')[-1]}".lower())).strip("-")

# Logging backend knob: "trackio" (default) or "none" for a fully offline/headless run.
REPORT_TO = os.environ.get("REPORT_TO", "trackio")
_report_kwargs = {"report_to": REPORT_TO}
if REPORT_TO == "trackio":
    _report_kwargs["trackio_space_id"] = GRPO_OUT

grpo_config = GRPOConfig(
    num_train_epochs=1,
    max_steps=GRPO_MAX_STEPS,
    learning_rate=1e-6,
    gradient_accumulation_steps=GRAD_ACCUM,
    # Keep at 1. TRL grows its environment pool only when a batch needs more
    # concurrent environment instances, and this env accepts a single session
    # at a time (see the "own environment Space" note at the top).
    per_device_train_batch_size=1,
    warmup_steps=min(10, GRPO_MAX_STEPS),
    optim="adamw_torch",
    max_grad_norm=1.0,
    num_generations=NUM_GENERATIONS,
    max_completion_length=1024,           # a whole multi-turn game, not one answer
    log_completions=True,
    num_completions_to_print=2,
    chat_template_kwargs=CHAT_TEMPLATE_KWARGS,   # shared with eval — see the knobs cell
    output_dir=GRPO_OUT,
    # Push weights to a repo DISTINCT from the Trackio Space id (=GRPO_OUT),
    # otherwise push_to_hub sees the Trackio-owned repo and skips as 'no files
    # modified'. A separate -model repo gets the actual checkpoint commit.
    hub_model_id=f"{HF_USERNAME}/{GRPO_OUT}-model" if HF_USERNAME else f"{GRPO_OUT}-model",
    logging_steps=1 if SMOKE else 10,
    save_strategy="no",
    gradient_checkpointing=True,
    **_report_kwargs,
    push_to_hub=not SMOKE,
    # vLLM OFF in-notebook (IPython init). The optional cell below enables it for an HF Job.
)

In [ ]:
# --- OPTIONAL: vLLM-accelerated GRPO rollouts (5-10x faster generation) ---------
# OFF by default: vLLM's init breaks under IPython/Jupyter, so leave USE_VLLM=0 for an
# in-notebook run. Enable it (USE_VLLM=1) ONLY in a non-IPython context: an HF Job
# or a fresh Colab runtime executed as a script. Colocate mode shares
# the single training GPU (right for Colab / one-GPU HF-Job flavors).
#   Verified TRL v1.7.0 params: use_vllm, vllm_mode="colocate"|"server",
#   vllm_gpu_memory_utilization. (server mode is multi-GPU; not used here.)
USE_VLLM = _env_flag("USE_VLLM", "0")
if USE_VLLM:
    grpo_config.use_vllm = True
    grpo_config.vllm_mode = "colocate"
    # Leave room for the training copy of the model in colocate mode; tune per GPU/model.
    grpo_config.vllm_gpu_memory_utilization = float(
        os.environ.get("VLLM_GPU_MEM_UTIL", "0.3")
    )
    print(
        f"[vLLM] enabled: mode=colocate gpu_mem_util={grpo_config.vllm_gpu_memory_utilization} "
        "(requires a non-IPython runtime + `pip install vllm`)"
    )
else:
    print("[vLLM] disabled (USE_VLLM=0). In-notebook generation uses HF generate().")

### Train the agent

`environment_factory=<EnvClass>` is the key line: for each rollout the trainer creates an env (TRL may reuse env instances across a batch), generates the model's response, parses its tool call, steps the env, and reads the reward — the agent loop, automated.

Here we train **directly from the base model** — no SFT warm-start — so GRPO learns purely from the environment reward.

> 💡 This is what makes it *agent* training rather than text fine-tuning: the data is generated by the policy acting in the env, not read from a file.

In [ ]:
import time

from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=MODEL_NAME,
    reward_funcs=reward_func,
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=WordleEnv,
)

# Wall-clock is the input to the calibration cell below — that is how the
# recommended full-run size gets measured instead of guessed.
_t0 = time.perf_counter()
trainer.train()
TRAIN_SECONDS = time.perf_counter() - _t0
print(f"training wall-clock: {TRAIN_SECONDS:.1f}s")

trainer.save_model(GRPO_OUT)
if not SMOKE:
    trainer.push_to_hub(commit_message="GRPO fine-tune on Wordle (textarena)")

### Calibrate the full run from this one

Rollout cost depends on the GPU, the policy size, and whether vLLM is on — so a
recommended step count that isn't measured on *your* hardware is just a guess. This
cell divides the training wall-clock by the rollouts actually consumed, then projects
the full run and reports the largest `GRPO_MAX_STEPS` that fits each `--timeout`.

> 💡 Run this under `SMOKE=1` first. The numbers it prints are what you feed into the
> `SMOKE=0` submission — `-e GRPO_MAX_STEPS=…` plus a matching `--flavor`/`--timeout`.

In [ ]:
def seconds_per_rollout(train_seconds: float, steps: int, grad_accum: int) -> float:
    """Measured cost of a single rollout.

    An optimizer step consumes `grad_accum` rollouts, so a run of `steps` steps
    consumes `steps * grad_accum` of them in total.
    """
    return train_seconds / max(1, steps * grad_accum)


def project_seconds(sec_per_rollout: float, steps: int, grad_accum: int) -> float:
    """Wall-clock a run of this size implies, at the measured per-rollout cost."""
    return sec_per_rollout * steps * grad_accum


def max_steps_within(budget_seconds: float, sec_per_rollout: float, grad_accum: int,
                     headroom: float = 0.75) -> int:
    """Largest step count fitting a timeout, leaving headroom for setup and eval."""
    return max(1, int((budget_seconds * headroom) / (sec_per_rollout * grad_accum)))


def _hms(seconds: float) -> str:
    return f"{seconds / 3600:.1f}h" if seconds >= 3600 else f"{seconds / 60:.1f}min"


sec_rollout = seconds_per_rollout(TRAIN_SECONDS, GRPO_MAX_STEPS, GRAD_ACCUM)
rollouts_done = GRPO_MAX_STEPS * GRAD_ACCUM

print(f"measured: {TRAIN_SECONDS:.1f}s over {rollouts_done} rollouts "
      f"-> {sec_rollout:.2f}s per rollout  (model={MODEL_NAME}, vram={vram_gb:.0f}GB, vllm={USE_VLLM})")

full_steps, full_ga = FULL_RUN["steps"], FULL_RUN["grad_accum"]
projected = project_seconds(sec_rollout, full_steps, full_ga)
print(f"\nprojected full run ({full_steps} steps x {full_ga} grad-accum "
      f"= {full_steps * full_ga:,} rollouts): {_hms(projected)}")

print("\nSteps that actually fit a given --timeout on THIS hardware:")
for budget in (3600, 10800, 21600, 43200):
    fits = max_steps_within(budget, sec_rollout, full_ga)
    print(f"  --timeout {budget:>6} ({_hms(budget):>5})  ->  GRPO_MAX_STEPS <= {fits}"
          f"   [{'full run fits' if projected <= budget else 'full run does NOT fit'}]")

print("\nRe-submit the full run with -e GRPO_MAX_STEPS=<value from the row you chose>.")
print("A bigger --flavor or -e USE_VLLM=1 lowers seconds-per-rollout; re-run this cell to re-measure.")

## 4 · Training signal — reward delta

A quick sanity check: compare mean reward over the first few logged steps against the
last few. This tells you whether reward *moved* during training — it is not a held-out
quality claim. The honest quality measure comes in §5, by playing full games.

In [ ]:
import statistics

rewards = [log["reward"] for log in trainer.state.log_history if "reward" in log]
if len(rewards) < 5:
    print(f"Only {len(rewards)} reward logs — increase max_steps for a clean delta.")
else:
    initial, final = statistics.mean(rewards[:5]), statistics.mean(rewards[-5:])
    print(f"initial={initial:.2%}  final={final:.2%}  delta={(final - initial) * 100:+.2f}pp")

## 5 · Test the trained agent — play full games

For a multi-turn agent the faithful test is **playing complete games**: the agent acts
turn by turn against live env feedback until the episode ends. We load the trained model,
let it guess turn-by-turn, and measure win rate over `N_EVAL_GAMES`.

**The prompt has to match training.** During training TRL renders each prompt with the
environment's tool schema attached; if evaluation renders it *without*, the policy is
being judged on a format it was never optimised for, and a lenient text parser will hide
that by still extracting a guess. So we pass `tools=env_tools(env)` — the same discovery
rule TRL uses — plus the same `CHAT_TEMPLATE_KWARGS`, and we report how many turns were
recovered by real tool calls versus by the fallback, so any residual mismatch is visible
in the output rather than silently absorbed.

In [ ]:
import json
import re
from collections import Counter

from transformers import AutoModelForCausalLM, AutoTokenizer


def parse_guess(text: str) -> tuple[str, str]:
    """Pull the guessed word out of one model turn.

    Returns `(word, how)` where `how` is one of:
      "tool_call" — a well-formed tool call, i.e. the format training used
      "regex"     — recovered from bare `[abcde]` text
      "raw"       — last-resort slice of the raw output

    The `how` tally matters: a large non-`tool_call` share means the eval prompt
    is not rendering the format the policy was trained on, and the win rate is
    measuring prompt mismatch rather than agent skill. Reporting it keeps that
    failure visible instead of silently absorbed by the fallback.
    """
    if "{" in text and "}" in text:
        try:
            payload = json.loads(text[text.index("{"): text.rindex("}") + 1])
            args = payload.get("arguments", payload)
            word = (args or {}).get("guess", "")
            if word:
                return word, "tool_call"
        except (ValueError, AttributeError):
            pass
    match = re.search(r"\[([a-zA-Z]{5})\]", text)
    if match:
        return match.group(1), "regex"
    return text.strip()[:5], "raw"


def play_one_game(env, model, tokenizer, verbose=False) -> tuple[bool, float, Counter]:
    """Play one Wordle game on an EXISTING env; return (won, reward, parse_tally).

    The env is created once by the caller and reused across games (`reset()` starts
    a fresh game). We do NOT close it here — closing the shared client mid-loop
    tears down the WebSocket the next game's `reset()` needs (ConnectionClosedOK).
    """
    obs = env.reset()
    messages = [{"role": "user", "content": WORDLE_PROMPT}]
    if obs:
        messages.append({"role": "user", "content": obs})
    how_tally: Counter = Counter()

    for _turn in range(6):
        if env.done:
            break
        # Same tool schema and same template kwargs the trainer used, so the model
        # sees at eval exactly the prompt format it was optimised on.
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tools=env_tools(env) or None,   # `or None`: an empty list still renders tool boilerplate
            add_generation_prompt=True,
            tokenize=False,
            **CHAT_TEMPLATE_KWARGS,
        )
        inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=512)
        text = tokenizer.decode(out[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
        if verbose:
            print(f"  model: {text[:120]}")

        word, how = parse_guess(text)
        how_tally[how] += 1
        try:
            feedback = env.guess(f"[{word}]")
        except Exception as exc:  # env rejected the move, or the game already ended
            if verbose:
                print(f"  env error: {exc}")
            break
        messages.append({"role": "assistant", "content": text})
        messages.append({"role": "user", "content": feedback})

    return (env.done and env.reward > 0), float(env.reward), how_tally

### Play full games with the trained agent

Wordle is multi-turn, so the faithful evaluation is to **play complete games**: the agent guesses, reads the env's letter feedback, and guesses again until it wins or runs out of turns. Win rate over several games is the agentic analogue of held-out accuracy.

> 💡 A single-shot accuracy number can't capture multi-turn behavior — you have to let the agent actually play.

In [ ]:
# Win rate of the trained agent over several games.
# One env is opened for the whole eval and reused across games (reset() per game),
# then closed once at the end — avoids tearing down the shared WebSocket mid-loop.
fine_tuned = AutoModelForCausalLM.from_pretrained(GRPO_OUT, dtype="auto", device_map="auto")
ft_tokenizer = AutoTokenizer.from_pretrained(GRPO_OUT)

eval_env = WordleEnv()
parse_modes: Counter = Counter()
try:
    wins, total_reward = 0, 0.0
    for g in range(N_EVAL_GAMES):
        won, r, how_tally = play_one_game(eval_env, fine_tuned, ft_tokenizer, verbose=(g == 0))
        wins += int(won)
        total_reward += r
        parse_modes += how_tally
        print(f"game {g + 1}/{N_EVAL_GAMES}: {'WON' if won else 'lost'}  reward={r:.2f}")
finally:
    try:
        eval_env.client.close()
    except Exception:
        pass

print(f"\nwin rate: {wins}/{N_EVAL_GAMES} = {wins / N_EVAL_GAMES:.0%}   mean reward={total_reward / N_EVAL_GAMES:.2f}")

# How the guesses were recovered. `tool_call` should dominate: it means the eval
# prompt rendered the same tool schema training used. A large `regex`/`raw` share
# would mean the win rate reflects prompt mismatch rather than agent skill.
turns = sum(parse_modes.values())
if turns:
    breakdown = "  ".join(f"{mode}={count} ({count / turns:.0%})" for mode, count in parse_modes.most_common())
    print(f"parse modes over {turns} turns: {breakdown}")

## Recap

**The workflow is the takeaway.** Agent RL is too long-running to babysit in an interactive
kernel, so we submitted the notebook itself to HF Jobs and let `papermill` execute it on
HF's cloud. The pieces that make that practical:

- **`SMOKE` gating** — every expensive quantity is an environment variable, so the same file
  is a five-minute proof or a real run with no cell edits.
- **Measured calibration** — the smoke run reports seconds per rollout on the actual GPU, and
  the full run's size is derived from that instead of assumed.
- **Compute auto-detection** — VRAM chooses the policy size, so one notebook runs unchanged
  across flavors.
- **Environment-Space ownership** — the env is single-session by design, so a training run
  needs your own duplicate, both to avoid contending with other users and to be able to
  restart it when a crashed run leaks its session.
- **Train/eval prompt parity** — the tool schema is derived from the env class by the same
  rule TRL uses, so evaluation measures the agent rather than a prompt mismatch.

The training payload itself is TRL's official Wordle example
([notebook](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb),
[guide](https://huggingface.co/docs/trl/en/openenv)) — that is deliberate: the point here is
what wraps around it.

This is the template for any OpenEnv environment: define the env wrapper (tools = public
documented methods), a reward function that reads `env.reward`, a dummy prompt dataset for the
episode count, and hand the class to `GRPOTrainer(environment_factory=...)`. Swap the env, keep
the spine — and because `env_tools()` names no tool, evaluation follows the swap for free.

**Where to go next:** the OpenEnv
[SFT-warmup tutorial](https://huggingface.co/docs/openenv/tutorials/sft-warmup) covers warm-starting
a policy with teacher rollouts before GRPO, and the
[end-to-end walkthrough](https://huggingface.co/docs/openenv/tutorials/end-to-end-walkthrough)
covers the single-turn case end to end.

## References

### This recipe builds on
- **TRL's official Wordle example**: [`openenv_wordle_grpo.ipynb`](https://github.com/huggingface/trl/blob/main/examples/notebooks/openenv_wordle_grpo.ipynb) — the system prompt, `WordleEnv` wrapper, reward function, GRPO config, and play-the-game eval follow it directly
- **TRL OpenEnv guide**: [huggingface.co/docs/trl/en/openenv](https://huggingface.co/docs/trl/en/openenv) — the `environment_factory` agent-training path
- **OpenEnv SFT-warmup tutorial**: [docs/openenv/tutorials/sft-warmup](https://huggingface.co/docs/openenv/tutorials/sft-warmup) — warm-starting a policy with teacher rollouts before GRPO
- **OpenEnv end-to-end walkthrough**: [docs/openenv/tutorials/end-to-end-walkthrough](https://huggingface.co/docs/openenv/tutorials/end-to-end-walkthrough) — the single-turn collect → train → evaluate pipeline

### Papers and Research
- **GRPO Algorithm**: [Group Relative Policy Optimization](https://huggingface.co/papers/2402.03300) — the original GRPO paper (DeepSeekMath), introducing value-free, group-relative policy optimization
- **TextArena**: [TextArena: A Framework for Text-based Game Environments](https://arxiv.org/abs/2504.11442) — the multi-turn text-game framework the Wordle environment is built on

### Libraries and Frameworks
- **TRL (Transformers Reinforcement Learning)**: [huggingface/trl](https://github.com/huggingface/trl) · [TRL GRPO docs](https://huggingface.co/docs/trl/en/grpo_trainer)
- **OpenEnv**: [huggingface/OpenEnv](https://github.com/huggingface/OpenEnv) — the environment client and async step/reset API · [OpenEnv announcement](https://huggingface.co/blog/openenv)
- **TextArena**: [LeonGuertler/TextArena](https://github.com/LeonGuertler/TextArena) — the underlying game engine served by the Wordle Space
- **Transformers**: [huggingface/transformers](https://github.com/huggingface/transformers) — `>=5.3.0` required for the tool-calling / chat-template behavior GRPO's env path depends on
- **Hugging Face Jobs**: [running Jobs from the CLI](https://huggingface.co/docs/huggingface_hub/en/guides/cli#hf-jobs) — non-interactive GPU execution on HF infrastructure
- **papermill**: [papermill docs](https://papermill.readthedocs.io/) — parameterized, non-interactive notebook execution
- **Trackio**: [gradio-app/trackio](https://github.com/gradio-app/trackio) — lightweight live training charts

### Environments and Models
- **Wordle environment Space**: [openenv/wordle](https://huggingface.co/spaces/openenv/wordle) — the hosted multi-turn OpenEnv/TextArena environment this notebook trains against
- **Qwen3-0.6B**: [Qwen/Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) — default policy on smaller GPUs
- **Qwen3-1.7B**: [Qwen/Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) — default policy when VRAM allows

### Key Concepts
- **Non-interactive notebook execution**: `papermill` runs the notebook as a batch job on HF Jobs, so hours-long agent RL survives a closed laptop and returns an executed notebook as its artifact
- **Measured calibration over assumed constants**: rollout cost is hardware- and model-dependent, so the smoke run measures it and the full run's size is derived rather than guessed
- **Multi-turn agentic RL**: one rollout spans several tool calls; the *environment* signals `done` (win or 6 guesses exhausted), so the trainer keeps stepping until the episode ends
- **Value-free RL (GRPO)**: no separate reward model — the environment's scalar reward is the only signal; rollouts are ranked *within* a sampled group
- **Stateful feedback slicing**: the env returns a cumulative transcript, so each turn we diff out only the newly appended feedback the agent should react to
- **Train/eval prompt parity**: the eval renders the same tool schema training used, derived from the env class by TRL's own discovery rule, so win rate measures the agent and not a format mismatch
- **Single-session environments**: OpenEnv environments not marked `SUPPORTS_CONCURRENT_SESSIONS` are capped at one session and the cap cannot be raised, which is why a training run needs your own Space duplicate